In [2]:
import torch
import torch.nn as nn
import pandas as pd

In [3]:
class AttentionBlock(nn.Module):
    def __init__(self, n_heads, d_model):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(4*d_model, d_model)
        )
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout = 0.15, batch_first = True)

    def forward(self, x, mask):
        x_normed = self.norm1(x)
        attention_output, _ = self.attention(x_normed,x_normed,x_normed, key_padding_mask = mask)
        x = x + attention_output
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
class Encoder(nn.Module):
    def __init__(self, n_heads, d_model, n_blocks, vocab_size, max_tokens):
        super().__init__()

        self.n_heads = n_heads
        self.d_model = d_model
        self.n_blocks = n_blocks
        self.vocab_size = vocab_size

        self.embeddings = nn.Embedding(self.vocab_size, d_model)
        self.positional_embedding = nn.Embedding(max_tokens, d_model)
        self.attention_blocks = nn.ModuleList([
            AttentionBlock(n_heads, d_model) for _ in range(self.n_blocks)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model /2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(d_model, 4)
        )

    def forward(self, x, attention_mask):
        B, T = x.shape
        mask = (self.attention_mask == 0)
        positions = torch.arange(T, device = x.device)
        x = self.embeddings(x) + self.positional_embedding(positions) #(B,T)->(B,T,C)

        for block in self.attention_blocks:
            x = block(x, mask)
        x = self.norm(x)
        x = self.classifier(x)
        return x
        

In [42]:
df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

In [43]:
df_train

,id,comment_text,toxic,severe_toxic,obscene,insult
0,691941fe3255065e,Christmas Is Where Family's Come Together To C...,1,0,1,1
1,c63d9e4b8c3417bd,"Im afraid you are wrong, i was never an admin ...",0,0,0,0
2,10a596707e7a5141,The orange infobox only has the hieroglyphs an...,0,0,0,0
3,16f740ee63e59ed0,"Completed item 1, when you're done with item 2...",0,0,0,0
4,b585008394bd8a2e,Contested deletion \n\nThis article should not...,0,0,0,0
...,...,...,...,...,...,...
39995,52d63df9facc5378,I really hope you faggots know how to swim bec...,1,0,0,0
39996,c771262430e24f6e,WHAT THE FUCK IS YOUR PROBLEM!!!!!!!!!!!!!!!!!...,1,1,1,1
39997,2dd3dc26fc66828a,Come to my house right now and do me in the bu...,0,0,1,0
39998,b0bf4747b5210024,"Sock-Puppetry \n\nFor the second time, this ti...",0,0,0,0


In [44]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-uncased"
)

In [45]:
example = tokenizer("hi im cool")

In [46]:
example

{'input_ids': [101, 7632, 10047, 4658, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}

In [47]:
padding_index = tokenizer.pad_token_id
vocab_size = tokenizer.vocab_size

In [48]:
from torch.utils.data import Dataset, DataLoader

In [49]:
class SentenceDataset(Dataset):
    def __init__(self, texts, y=None):
        self.texts = texts
        self.y = y
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, index):
        text = self.texts[index]
        if self.y is not None:
            labels = self.y[index]
        else:
            labels = (0,0,0,0)

        return text, labels

In [50]:
train_data = SentenceDataset(df_train['comment_text'], [(x['toxic'], x['severe_toxic'], x['obscene'], x['insult']) for _,x in df_train.iterrows()])

In [51]:
test_data = SentenceDataset(df_test['comment_text'])

In [ ]:
def collate_fn(batch):
    texts = []
    labels = []
    for x,y in batch:
        texts.append(x)
        labels.append(torch.tensor(y))

    output = tokenizer(texts, max_length=128, padding=True, truncation=True, return_tensors="pt")

    return output['input_ids'], output['attention_mask'], labels

In [37]:
train_dataloader = DataLoader(train_data, batch_size= 64, collate_fn=collate_fn, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size= 64, collate_fn=collate_fn, shuffle=False)

In [ ]:
model = Encoder(4,128,3,vocab_size,128)
optimizer = torch.optim.AdamW(params=model.parameters(),lr = 1e-4, weight_decay=1e-3)
epochs = 5